# Workshop 4: Open-TYNDP Outcomes and CBA Workflows

:::{note} At the end of this notebook, you will be able to:

**1. Inspect Open-TYNDP benchmarks**
  - Navigate and analyze Open-TYNDP NT scenario outcomes using PyPSA's statistics module
  - Investigate latest networks using PyPSA-Explorer
  - Compare your findings with the benchmarking framework
  
**2. Run CBA workflows**
  - Modify the CBA configuration for the targeted project and/or climate year
  - Learn how to execute CBA analysis coupled to or detached from the Scenario Building workflow

:::

:::{note}
If you have not set up Python on your computer, you can execute this tutorial in your browser via [Google Colab](https://colab.research.google.com/). Click the rocket button in the top right corner and launch "Colab". If that doesn't work, download the `.ipynb` file and import it in [Google Colab](https://colab.research.google.com/).

Then install the required packages by uncommenting the cell below.

:::

In [ ]:
# uncomment for running this notebook on Colab
# !pip install pypsa==1.2.1 pypsa-explorer pandas matplotlib numpy pdf2image cartopy
# !apt-get install poppler-utils

In [ ]:
import os
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd
import pypsa
from IPython.display import SVG, Code, IFrame, Image, display
from pdf2image import convert_from_path
from pdf2image.exceptions import PDFPageCountError
from pypsa.plot.maps.static import add_legend_lines
from pypsa_explorer import create_app

pypsa.set_option("params.statistics.nice_names", True)
pypsa.set_option("params.statistics.drop_zero", True)
pypsa.set_option("params.statistics.round", 3)
plt.rcParams["figure.figsize"] = [14, 7]
clip = 1  # TWh

In [ ]:
def unzip_with_timestamps(zip_path, extract_to, keep_zip=True):
    """Unzip a file while preserving original file timestamps."""
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        for member in zip_ref.infolist():
            # Extract the file
            zip_ref.extract(member, extract_to)

            # Get the extracted file path
            extracted_path = os.path.join(extract_to, member.filename)

            # Get the modification time from the zip file
            date_time = datetime(*member.date_time)
            timestamp = date_time.timestamp()

            # Set both access and modification times
            os.utime(extracted_path, (timestamp, timestamp))
    if not keep_zip:
        os.remove(zip_path)

As in previous workshops, we first need to retrieve some files that we'll need today.

In [ ]:
urls = {
    "data/results-0.7.1.zip": "https://storage.googleapis.com/open-tyndp-data-store/outcomes/0.7.1/results-0.7.1.zip",
    "scripts/_helpers.py": "https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/refs/heads/feat/workshop-4/open-tyndp-workshops/scripts/_helpers.py",
}

os.makedirs("data", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
for name, url in urls.items():
    if os.path.exists(name):
        print(f"File {name} already exists. Skipping download.")
    else:
        print(f"Retrieving {name} from storage.")
        urlretrieve(url, name)
        print(f"File available in {name}.")

to_dir = "data/results-0.7.1"
if not os.path.exists(to_dir):
    print(f"Unzipping data/results-0.7.1.zip.")
    unzip_with_timestamps("data/results-0.7.1.zip", "data/results-0.7.1")
print(f"Latest NT results for Open-TYNDP v0.7.1 are available in '{to_dir}'.")

print("Done")

And we'll also import some handy helper functions that we introduced in the last workshops.

In [ ]:
from scripts._helpers import (
    display_code_lines,
    run_pypsa_explorer_in_colab,
    show_benchmarks,
)

# Scenario Building

## Explore NT outcomes using the ``PyPSA.statistics`` module

`n.statistics` provides a consistent, high-level API that handles component iteration, port mapping, and carrier grouping automatically.

:::{tip}

`n.stats` is available as a shorthand alias for `n.statistics`.

:::

Each method can be called individually or explored via the summary table:

| Category | Methods |
|---|---|
| **Costs** | `capex()`, `installed_capex()`, `expanded_capex()`, `opex()`, `system_cost()` |
| **Capacity** | `installed_capacity()`, `optimal_capacity()`, `expanded_capacity()`, `capacity_factor()` |
| **Energy** | `supply()`, `withdrawal()`, `energy_balance()`, `transmission()`, `curtailment()` |
| **Market** | `prices()`, `revenue()`, `market_value()` |

Every method accepts the same filtering and grouping parameters:
| Parameter | Description |
|---|---|
| `groupby` | String, list, or callable — how to group results (default: `\"carrier\"`) |
| `groupby_method` | Aggregation function (`\"sum\"` (default), `\"mean\"`, …) |
| `groupby_time` | `\"sum\"`, `\"mean\"`, or `False` for time series — default varies by method |
| `components` | Filter to specific component types |
| `carrier` | Filter by carrier name (internal name) |
| `bus_carrier` | Filter by the carrier of the bus |
| `nice_names` | Use human-readable carrier names (default: `True`) |

:::{warning}

`prices()` has a simplified interface — `groupby` and `groupby_time` are booleans,
and it does not accept `carrier` or `components`.

:::

The full `PyPSA.statistics` API documentation is available in the [pypsa documentation](https://docs.pypsa.org/latest/user-guide/statistics/). Additionally, you can find two video tutorials on *PyPSA meets Earth*'s youtube channel ([part 1](https://www.youtube.com/watch?v=_Asjhz6We5o), [part 2](https://www.youtube.com/watch?v=nlh3azf2JJw)) for more comprehensive information and examples on how to use the statistics module. This learning material is open-source and available on [GitHub](https://github.com/open-energy-transition/pypsa-training-sessions).

### Reminder: Extracting insights from the network

Let's load the latest Open-TYNDP NT scenario outcomes (v0.7.1) again and explore them using the statistics module.

In [ ]:
# Load latest NT scenario networks
base_path = "data/results-0.7.1/NT-cy2009-20260520/networks/"


def import_network(fn: str):
    n = pypsa.Network(fn)
    n.carriers.loc["none", "color"] = "#000000"
    return n


# Load networks directly into dictionary for PyPSA-Explorer
networks = {
    "NT 2030": import_network(base_path + "base_s_all___2030.nc"),
    "NT 2040": import_network(base_path + "base_s_all___2040.nc"),
}

To start of we'll have a look at the network for the `NT 2030` scenario.

In [ ]:
scenario = "NT 2030"


For convenience, let's save the statistics accessor in a variable `s`.

In [ ]:
s2030 = networks[scenario].stats

You can easily get a comprehensive overview of all system-level metrics at once.

In [ ]:
s2030()

Of course, this can be a bit difficult to grasp. So let's have a look at some specific metrics instead, for example installed renewable capacities:

In [ ]:
(
    s2030.installed_capacity(
        bus_carrier="AC|low voltage",
        comps="Generator",
    )
    .div(1e3)  # GW
    .round(3)
    .to_frame("Installed Capacity [GW]")
    .filter(regex="^(?!Demand).*", axis=0)
)

We can also easily look into the outputs of the model.

So, let's investigate electricity supply and demand for our NT 2030 network using the `energy_balance` method:

In [ ]:
balance = (
    s2030.energy_balance(
        bus_carrier=["AC", "low voltage"],
        comps=["Generator", "Link", "StorageUnit", "Load"],
        groupby=["carrier"],
        aggregate_across_components=True,
    ).div(
        1e6
    )  # TWh
    # .sort_values(ascending=True)
)

# Format output
balance = balance[
    abs(balance.values) > clip
].to_frame(  # Filter for entries > clipped value
    "Supply (+), Demand (-) [TWh]"
)
balance.style.format("{:,.2f}")  # Make style a bit prettier

We can also visualize this for a better overview:

In [ ]:
balance.plot.barh(
    figsize=(5, 12),
    xlabel="TWh",
    title=f"Electricity Energy Balance {scenario} (clipped at {clip} TWh)",
);

### Reminder: Visualizing results using `PyPSA.statistics`

As we already introduced in a previous workshop, it is actually much quicker to explore the data with plots generated using the `PyPSA.statistics` module.

For example the electricity energy balance we investigated before...

In [ ]:
fig, ax, _ = s2030.energy_balance.plot.bar(
    bus_carrier=["AC", "low voltage"],
    query=f"abs(value)>{clip * 1e6}",
    height=6,
)
ax.set_title(f"Electricity Energy Balance {scenario} (Clipped at {clip} TWh)");

...or we can even explore time series interactively:

In [ ]:
s2030.energy_balance.iplot.area(
    bus_carrier=["AC", "low voltage"],
    y="value",
    x="snapshot",
    color="carrier",
    stacked=True,
    width=1800,
    height=600,
    query="snapshot < '2009-03'",
    title="Electricity Energy Balance NT 2030, January - February",
)

We can easily have a closer look at the production of a specific technology and a specific country. For example January wind production in the Netherlands in NT 2030.

In [ ]:
fig = s2030.energy_balance.iplot.area(
    facet_col="country",
    y="value",
    x="snapshot",
    carrier="wind",
    color="carrier",
    query="country == 'NL' and snapshot < '2009-02'",
    width=1800,
    height=500,
    title="Wind Production Netherlands NT 2030, January",
)

fig.update_layout(yaxis_title="Wind Production [MWh_el]")

As you can see, in January the Netherland's wind mix is largely dominated by Offshore Wind production.

### Compare results with ``PyPSA.NetworkCollections``

Lastly, one might also want to compare results between runs or planning years. To make this a bit easier, PyPSA v1.0 introduced a new object called `NetworkCollections`.

Let's have a look at how this is used in practice. First we define a network collection (`nc`) with our previously imported result networks for NT 2030 and NT 2040.

In [ ]:
nc = pypsa.NetworkCollection(networks)
nc

As we can see, our `NetworkCollection` contains two networks, the `NT 2030` network and the `NT 2040` network.

We can now use `PyPSA.statistics` accessor directly on this `NetworkCollection` instead of a single network to get the metrics for them simultaneously. 

Let's start again by defining a helper variable `sc` for the statistics accessor to make our life a bit easier going forward.

In [ ]:
sc = nc.statistics

Now, let's have a look at the electricity energy balance again but for both networks at once:

In [ ]:
ebs = (
    sc.energy_balance(
        groupby=["bus_carrier", "carrier"],
        aggregate_across_components=True,
    )
    .div(1e6)
    .unstack("network")
    .loc[["Electricity", "Electricity Low Voltage"]]
    .droplevel("bus_carrier")
)

# Format output
ebs = ebs[abs(ebs["NT 2030"]) > clip]  # Filter for entries > 1 TWh
ebs.style.format("{:,.2f}")  # Make style a bit prettier

We can again create a simple plot to make this a bit easier to investigate

In [ ]:
ebs.plot.barh(
    figsize=(6, 10),
    xlabel="Supply (+), Demand (-) [TWh]",
    title=f"Electricity Energy Balance NT (clipped at {clip} TWh)",
);

Similarly, we can also look into electricity prices in the system. 

Note that you can easily decide if you want to calculate the price weighted by load instead using `weighting='load'`.

In [ ]:
prices = sc.prices(bus_carrier="AC", groupby=False, weighting="time").unstack("network")
prices.head(10)

Again we plot them for easier readability

In [ ]:
prices.plot.bar(
    figsize=(25, 4),
    edgecolor="white",
    ylabel="€/MWh",
    xlabel="Bus Carrier",
    title="Electricity Price by node",
);

:::{Note}

Please keep in mind that the methodology used to implement hydrogen and electricity market coupling slightly differs from the TYNDP 2024 approach. Unlike the Market Model, which assumes a fixed hydrogen fuel price for hydrogen-to-power generation, Open-TYNDP couples electricity and hydrogen markets by using endogenous hydrogen fuel price for them. This results in high price spikes induced by load shedding in the coupled market. This is especially visible in the NT 2040 outcomes. Detailed electricity price benchmarks excluding load shedding are available on [Zenodo](https://zenodo.org/records/20303009).

:::

Of course `NetworkCollections` also work with `PyPSA.statistics` quick plotting API:

In [ ]:
fig = sc.prices.iplot.bar(
    bus_carrier=["AC"],
    x="name",
    y="value",
    color="network",
    stacked=True,
    width=1800,
    height=500,
    nice_names=False,
    title="Electricity Price by node",
)

fig.update_layout(yaxis_title="Prices [EUR/MWh]")

## Task 1: Grow comfortable with ``PyPSA.statistics``
Familiarize yourself with the statistics module (again) and explore the latest outcomes of Open-TYNDP using the different methods and plots introduced today.

**Hint**: You can also refer to the [introduction](#reminder-the-pypsa-statistics-module) above for more information on the different methods and parameters of ``PyPSA.statistics``.

## Task 2: Reproduce Benchmarks
(a) If you feel comfortable using `PyPSA.statistics`, you can try to reproduce the Open-TYNDP outcomes from the following example of our latest [benchmarking figures](https://zenodo.org/records/20303009).

Try it without looking at the previous example first.

In [ ]:
show_benchmarks(
    "benchmark_hydrogen_price_cy2009",
    [2030],
    "data/results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

(b) Optional: Try to exclude load shedding from the hydrogen price in 2040.

## Task 3 (Advanced): Inspect Outputs

(a) Can you verify the amount of wind generated on the Bornholm Energy Island in 2040 at *10.8 TWh*?

**Hint:** The Offshore Hub bus carrier is `AC_OH` and Borholm Energy Island bus is called `BEIOH01`.

(b) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

**Hint:** Look for `H2 pipeline` in the energy balance.

(c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?

## Interactive Exploration with PyPSA-Explorer

As we introduced in the last workshop, PyPSA-Explorer is an interactive dashboard for visualizing and analyzing energy system networks. It provides:
- Energy balance analysis with both time series and aggregated views
- Capacity planning visualizations by technology and region
- Economic analysis showing CAPEX/OPEX breakdowns
- Interactive geographical network maps
- Support for visualising multiple networks

### Reminder: Using the Dashboard

Once the dashboard opens, you can explore these key features:

**1. Energy Balance Tab**
   - View production, consumption, and storage patterns over time
   - Switch between time series and aggregated views
   - Filter by energy carrier (electricity, hydrogen, etc.)
   - Filter by country or region

**2. Capacity Tab**
   - Analyze installed capacities across scenarios
   - Compare capacity buildout between 2030 and 2040
   - View breakdowns by technology type and region

**3. Economics Tab**
   - Examine costs and revenues
   - Review CAPEX and OPEX breakdowns by technology
   - Compare regional cost distributions
   - Assess investment requirements

**4. Network Map**
   - Visualize the geographical network layout
   - View an interactive map with network components
   - Zoom and pan to explore specific regions

**Tip:** Use the scenario selector buttons in the top-right corner to switch between NT 2030 and NT 2040 scenarios.

:::{note}

 PyPSA-Explorer can be launched in different ways depending on your environment:

- **Local Jupyter**: Use the terminal command (recommended) or inline display
- **Google Colab**: The dashboard launches inline, embedded directly in the notebook

Follow the instructions below for your specific environment.

:::

In [ ]:
# Detect if running on Google Colab
try:
    from google.colab import output

    IN_COLAB = True
    print(f"This notebook is running on Google Colab!")
except ImportError:
    IN_COLAB = False
    print(f"This notebook is running locally !")

port = 8050

#### For Local Users

If you're running locally, we **recommend** launching PyPSA-Explorer from the terminal for optimal performance:

```bash
pypsa-explorer data/results-0.7.1/NT-cy2009-20260520/networks/base_s_all___2030.nc:NT_2030 data/results-0.7.1/NT-cy2009-20260520/networks/base_s_all___2040.nc:NT_2040
```

This command opens the dashboard in your default browser at http://localhost:8050.

**Alternative**: The cell below can launch the dashboard inline within the notebook, though the terminal method provides better performance and responsiveness.

In [ ]:
# Terminal method recommended
USE_TERMINAL = True  # Change to False if you want to launch from the notebook instead

if not IN_COLAB and not USE_TERMINAL:
    # Local Jupyter: Inline display
    app = create_app(networks)
    app.run(jupyter_mode="tab", port=port, debug=False)

#### For Google Colab Users

Running PyPSA-Explorer on Google Colab requires a small workaround to display the dashboard properly inside the notebook.

We already imported a useful helper function we introduced in the last workshop to handle this: `run_pypsa_explorer_in_colab()`

In [ ]:
if IN_COLAB:
    run_pypsa_explorer_in_colab(networks, port)

**Tip for Colab users:** To view the dashboard in fullscreen mode, click the three dots (⋮) in the top-right corner of the output cell and select **"View output fullscreen"**.

## Task 4: Navigate ``PyPSA-Explorer``
Let's look at the latest Open-TYNDP NT scenario results (v0.7.1) again and explore them interactively using the PyPSA-Explorer. Try again to find some specific insights we manually extracted in the previous section.

(a) Can you verify the amount of wind generated on the Bornholm Energy Island in 2040 at *10.8 TWh*?

(a) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

(c) Can you verify the observed correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040?

# Cost-Benefit Analysis

Now that we've explored the Scenario Building (SB) results, let's learn how to run Cost-Benefit Analysis (CBA)!

## Workflow management using `Snakemake` and `pixi`

The Open-TYNDP CBA workflow involves many interconnected steps: from retrieving the SB network, to preparing the reference and project networks, to optimizing the networks, to calculating indicators. 

To manage this complexity, Open-TYNDP uses two complementary tools:

1. **`Snakemake`** - A workflow management system that automatically figures out which analysis steps to run and in what order
2. **`pixi`** - A package manager that simplifies environment setup and provides easy-to-use shortcuts for running workflows

The combination of `Snakemake` and `pixi` allow Open-TYNDP to run with the flexibility to easily change configurations and run different scenarios. 

### Reminder: `Snakemake`

<img src="https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/792b8474ab5096e5ab8db2822af4fcd9fe659eb6/open-tyndp-workshops/snakemake_logo.png" width="500px" />

The `Snakemake` workflow management system is a tool to create reproducible and scalable data analyses.
Workflows are described via a human readable, Python based language. They can be seamlessly scaled to server, cluster, grid, and cloud environments, without the need to modify the workflow definition.

Snakemake follows the [GNU Make](https://www.gnu.org/software/make) paradigm: workflows are defined in terms of so-called `rules` that specify how to create a set of output files from a set of input files. Dependencies between the rules are determined automatically, creating a DAG (directed acyclic graph) of jobs that can be automatically parallelized.

**Why does Open-TYNDP use Snakemake?**

Running the full TYNDP analysis involves many steps that depend on each other. 

Snakemake can automatically:

- Determine which steps need to run based on what files already exist
- Figure out the correct order to run them
- Skip steps that don't need to be re-run
- Can run independent steps in parallel to save time

:::{note}

Snakemake documentation: https://snakemake.readthedocs.io/

:::

#### Using `snakemake`

You can trigger the workflow by specifying a target file, like `data/benchmark.pdf`, or any intermediate file:
```bash
snakemake -call data/benchmark.pdf
```

Alternatively, you can also execute the workflow by calling a rule that produces an intermediate file:
```bash
snakemake -call build_data
```
**NOTE:** You cannot call a rule that includes a wildcard without specifying what the wildcard should be filled with. Otherwise, Snakemake will not know what to propagate back.

Or you can call the common rule `all` which can be used to execute the entire workflow. It takes the final workflow output as its input and thus requires all previous dependent rules to be run as well:
```bash
snakemake -call all
```

Because we defined the `all` rule as first in the `Snakefile`, this rule is assumed to be the default and the following also works:
```bash
snakemake -call
```

A very important feature is the `-n` flag which executes a `dry-run`. It is recommended to always first execute a `dry-run` before the actual execution of a workflow. This simply prints out the DAG of the workflow to investigate without actually executing it.

Let's try this out and investigate the output:

In [ ]:
! snakemake -call -n

### Introducing: `pixi`

<img src="https://raw.githubusercontent.com/prefix-dev/pixi/9f94b8a24353ca88d21846b72438c6066e8adc74/docs/assets/pixi-logo.svg" width="150px" />



`pixi` is a cross-platform, multi-language package management and workflow tool. It is built on the foundation of the `conda` ecosystem.

**Why does Open-TYNDP use `pixi`?**

`pixi` serves two important roles in the Open-TYNDP project:

**1. Environment Management**  
`pixi` automatically installs all the required Python packages and their correct versions, similar to `conda` but faster and more reliable. Pixi helps us not have to worry about package conflicts or missing dependencies.

**2. Simplified Commands**  
`pixi` also allows us to create shortcuts for long snakemake commands. 

You can see the full Snakemake commands that pixi runs by looking in the `pixi.toml` file in the Open-TYNDP repository. For example, we have defined a shortcut for running the full CBA workflow, called ``tyndp-cba``.

:::{note}

`pixi` documentation: https://pixi.prefix.dev/latest/.

:::


#### Using `pixi`

**Without pixi (raw Snakemake), you would run this command to run the full CBA workflow:**
```bash
snakemake -call cba --configfile config/config.tyndp.yaml
```

**Using `pixi`, you just need to run:**
```bash
pixi run tyndp-cba
```

For the remainder of this notebook, we will use `pixi` commands. 


## Coupled vs decoupled SB-CBA workflow

<center>

![](cba_multi_proj.png)
![](cba_from_presolved.png)

</center>

The two diagrams above illustrate the two different ways to run the CBA workflow:

**Coupled Workflow (left):**
- Starts from scratch with the Scenario Building optimization
- First solves the Scenario Building network (the `solve_sector_network_myopic` step)
- Then continues on to perform the Cost-Benefit Analysis
- Each CBA project goes through: prepare project -> optimize project and reference networks -> calculate indicators

**Note:** There are many rules (100+) that exist before `solve_sector_network_myopic`, but they are not included here for the diagram to be legible. With these diagrams, we mainly want to highlight the handoff between the SB process and the CBA process within the Open-TYNDP workflow.

**Note:** For each additional project you evaluate, the workflow adds more parallel branches. The diagram shows 2 projects just for illustrative purposes.

**Decoupled Workflow (right):**
- Starts from pre-solved SB networks that we download from our releases
- Skips all the Scenario Building steps -- begins directly at the `retrieve` stage
- The rest of the workflow is identical to the coupled approach
- Much faster because we can skip the Scenario Building steps (including the optimization)

## Cloning and installing Open-TYNDP

In [ ]:
# Uncomment the next line for running this notebook on Colab
# !wget -qO- https://pixi.sh/install.sh | sh

First, navigate into the `data/` folder:

In [ ]:
os.chdir("data/")
print("Directory changed to:", os.getcwd())

Clone the Open-TYNDP repository directly:

In [ ]:
if os.path.basename(os.getcwd()) == "data":
    if not os.path.exists("open-tyndp"):
        ! git clone https://github.com/open-energy-transition/open-tyndp.git
    else:
        print("open-tyndp directory already exists, skipping clone")
else:
    print("Warning: Not in data/ directory. Current directory:", os.getcwd())

Now, navigate into the the `open-tyndp` directory:

In [ ]:
os.chdir("open-tyndp/")
print("Directory changed to:", os.getcwd())

Use `pixi` to install the `open-tyndp` environment:

In [ ]:
! pixi install -e open-tyndp

## Running a coupled SB->CBA workflow

As mentioned earlier, the command `pixi run tyndp-cba` executes the complete CBA workflow:
1. Takes a solved Scenario Building network (either from scratch or using pre-solved network)
2. Prepares the CBA reference and project networks, for the specified project(s)
3. Evaluates each specified project using TOOT or PINT methodology
4. Calculates indicators (B1-B4)

**Configuration Options**

The Open-TYNDP workflow's settings are housed in `config/config.tyndp.yaml`. Feel free to open the file to explore the full extent of configuration settings we have available.

One key parameter that affects both the SB and CBA processes is the run name:

Run name:
```yaml
run:
  name: "all" # which scenario to run - for example, can be changed to "NT" to run just the NT scenario
```

Some CBA-specific key parameters are:

```yaml
cba:
  planning_horizons: # which planning horizon(s) to assess project on
  - 2030
  - 2040
  projects:
  - t1-t35 # which projects to assess
```

You can modify these settings in two ways:
1. **Edit the config file directly**
2. **Override via command line** -- For example, by adding the following to your command: `--config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030]}'`

:::{tip}
In practice, editing the YAML configuration files directly in a text editor or IDE is much easier than using command-line overrides. We're using command-line options in this notebook for demonstration purposes, but for real life, we recommend modifying the config files directly!
:::

First, let's check what running the full workflow could look like by doing a dry-run of `pixi run tyndp-cba`:

In [ ]:
! pixi run tyndp-cba -n

We can specify the specific scenario (e.g, `NT`), the project (e.g., `t4`), and planning horizon (e.g., 2030) we want to run directly in the command line.

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Notice how the `count` of steps changed when we specified a single run, project, and horizon.

Comparatively, the default settings in Open-TYNDP is set to run:
- NT, DE, GA, and all climate year variations of these scenarios
- two planning horizons (2030 and 2040)
- 5 CBA projects (t4, t16, t28, t33, t35)

Thus, running the full default CBA workflow will trigger many, many more steps, since the DAG has to be expanded for all scenarios, planning horizons, and projects.

### Checkpoints

You may notice above that the number of rules is actually quite low.

There is a `checkpoint` in the workflow, called `clean_projects`, that first checks how many projects are being asked to run before building out the full DAG.
It tells the workflow which CBA projects exist, which project IDs to run, and which method applies (TOOT/PINT).

Before the `clean_projects` step runs, Snakemake does not know the full list of project jobs it needs to create.
The step downloads and cleans the external CBA project database, tells the workflow the full list of projects available, which projects the user wants to evaluate, and how each should be evaluated.
After `clean_projects` finishes, Snakemake can read the cleaned CSV and expand the DAG into concrete jobs.

Thus, what we should do first here is run the workflow just up until the checkpoint, then only after that can we run the full workflow again -- in which case, the DAG would show the actual number of jobs that would be run.

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

Now that the checkpoint is complete, we can re-check the DAG of how many steps are needed to run the CBA workflow.

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

### Task 5: Configure the settings to run the NT scenario for a single CBA project and planning horizon

So far, we've been setting the run settings via the command line. However, as we've mentioned, in real life it's much easier to go into the config files and edit them directly. So let's do that for the settings we've been running the CBA with so far:
- NT run
- 2030 planning horizon
- t4 project

**Instructions:**
1. Open `config/scenarios.tyndp.yaml` 
2. Find the `NT` scenario definition (it's the first one)
3. Modify the `NT` scenario so that it looks like this:

```yaml
NT:
  tyndp_scenario: NT

  cba:
    planning_horizons: [2030]
    projects: ["t4"]
```

4. Save the file
5. Open `config/config.tyndp.yaml` in a text editor
6. Set `run: name:` to `"NT"`. The top section of your file should look like:

```yaml
run:
  prefix: "tyndp"
  name: "NT"
```

7. Save the file
8. Come back to this notebook to run the checkpoint: `! pixi run tyndp-checkpoint`
9.  Then run the dry-run: `! pixi run tyndp-cba -n`

You should see the same number of steps as when we ran `! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n` above.

Now you can run the CBA workflow without needing to specify configuration overrides on the command line!

## Modifying climate years

**Understanding Climate Years**

Energy system performance varies significantly with weather conditions - wind and solar availability change year to year, as do heating and cooling demands. To account for this variability, Open-TYNDP can evaluate scenarios using different historical climate years.

Climate year scenarios are defined in `config/scenarios.tyndp.yaml`. For example:

```yaml
NT-cy2008:
#  <<: *cba-common
  snapshots:
    start: "2008-01-01"
    end: "2009-01-01"

  atlite:
    default_cutout: europe-2008-sarah3-era5

  cba:
    sb_scenario: NT
```

The following climate year scenarios related to the `NT` scenario are already existing in the Open-TYNDP workflow:
- `NT-cy1995`: NT scenario using 1995 weather data
- `NT-cy2008`: NT scenario using 2008 weather data  
- `NT-cy2009`: NT scenario using 2009 weather data
- `NT-cyears`: Runs all 3 NT scenarios: `NT-cy1995`, `NT-cy2008`, and `NT-cy2009`

The `NY-cyears` scenario acts somewhat as a collection scenario for the 3 climate years and is defined in `config/scenarios.tyndp.yaml` as well:

```yaml
NT-cyears:
  cba:
    scenarios: [NT-cy2009, NT-cy2008, NT-cy1995]
```

How can we run the CBA workflow for a different climate year? First, we again need to run up to the checkpoint for the different climate years:

In [ ]:
! pixi run tyndp-checkpoint --config 'run={"name":["NT-cy2008", "NT-cy2009", "NT-cy1995"]}' 'cba={"planning_horizons":[2030],"projects":["t4"]}'

Now, we can run for just another climate year, such as the 1995 climate year for the NT scenario (`NT-cy1995`):

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT-cy1995"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Or, we can run all climate years (1995, 2008, and 2009), which falls under the `NT-cyears` scenario:

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT-cyears"}' 'cba={"planning_horizons":[2030],"projects":["t4"]}' -n

Notice how the total count of steps is much higher (especially for CBA-specific steps such as `make_indicators` and weather-dependent steps such as `build_renewable_profiles_pecd`) when multiple climate years are run.

### Task 6: Create your own climate year and run it

Create your own custom climate year scenario by modifying `config/scenarios.tyndp.yaml`:

**Instructions:**
1. Open `config/scenarios.tyndp.yaml` 
2. Find an existing climate year definition (e.g., `NT-cy2008`)
3. Copy the entire section and give it a new name (e.g., `NT-workshop` or anything you like)
4. Modify the parameters as desired (e.g., set the snapshots to the climate year you're interested in - note that we're limited to 1995, 2008, and 2009)
5. Add CBA configuration to your custom scenario (similar to what you did in Task 5):
```yaml
   cba:
    sb_scenario: NT
    planning_horizons: [2030]
    projects: ["t4"] 
```

In the end, the full custom scenario should look something like this:

```yaml
NT-workshop:
#  <<: *cba-common
  snapshots:
    start: "2008-01-01"
    end: "2009-01-01"

  atlite:
    default_cutout: europe-2008-sarah3-era5

  cba:
    sb_scenario: NT
    planning_horizons: [2030]
    projects: ["t4"] 
```

6. Save the file
7. Update `config/config.tyndp.yaml` to set run: name: to `["NT-workshop"]` (or your custom scenario name)

By configuring the CBA settings in your scenario file, you can now run it with just: `pixi run tyndp-cba --config 'run={"name":["NT-workshop"]}' -n` (first, remember to run the checkpoint).

:::{warning}
The weather data availability is limited. Currently, only climate years 1995, 2008, and 2009 are available in the PECD (Pan-European Climate Database) that Open-TYNDP uses.
:::

## Running a decoupled SB->CBA workflow


You may have noticed by now that when running the coupled SB and CBA workflow, even when running the CBA for just 1 project, 1 planning horizon, and 1 scenario, this could trigger 100+ steps to be run.

Within Open-TYNDP, we have the flexibility to retrieve a pre-solved SB network (uploaded to Zenodo and Google Cloud as part of the project releases) and use this pre-solved SB network



This can be easily done by setting the `cba.use_presolved` setting to `true`. By default, this downloads and uses the latest release we have (at the time of this workshop, that is v0.7.1).

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT"]}' 'cba={"cba_scenario_input":{"use_presolved":true,"sb_version":"latest"},"planning_horizons":[2030],"projects":["t4"]}' -n

Note how the number of steps has decreased down from over 100 to just ~37, and we see some new steps not previously seen before, such as `retrieve_presolved_sb_networks`.

You can also specify a different version to retrieve from - for example, the previous release (0.6.1):

In [ ]:
! pixi run tyndp-cba --config 'run={"name":["NT"]}' 'cba={"cba_scenario_input":{"use_presolved":true,"sb_version":"0.6.1"},"planning_horizons":[2030],"projects":["t4"]}' -n

:::{warning}:::
At the moment, Open-TYNDP has only run the SB process on the NT scenario, so we can only use pre-solved networks for the NT scenario (meaning, you cannot set `use_presolved` to `true` and change the run name to "DE" for example). 

#### Task 7 (optional): Perform a complete Cost-Benefit Analysis on a project (decoupled)

**Optional Exercise: Run a Complete CBA**

If you have time and want to see the full workflow in action, you can run a complete CBA evaluation using a pre-solved network:

First, navigate into a specific branch called `workshop-4-cba` in git:

```bash
! git checkout workshop-4-cba
```

We created this branch to allow for a successful lower-resolution optimization. 

The only changes made to this branch are changes to `config.tyndp.yaml`:
- retrieve data from Google Cloud (instead of Zenodo)
- remove hydro-reservoir state of charge constraint
- allow load sinks for CO2 sequestration

The first adjustment is just to speed up retrieval times, while the last two adjustments are needed mainly to alleviate constraint conflicts from the lower temporal resolution of used in the optimization for this workshop (otherwise, we normally solve at 1H resolution, which takes too long for such an exercise). The changes made can also easily be seen in the Github Pull Request associated with the branch: https://github.com/open-energy-transition/open-tyndp/pull/732/changes. 

After checking out the branch, you can run the following command:

```bash
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'data_config=tyndp' 'cba={"planning_horizons":[2030],"cba_scenario_input":{"use_presolved":true,"sb_version":"latest"},"projects":["t16"],"msv_extraction":{"resolution":"24H"}}'
```

**What this command does:**
- Downloads the pre-solved NT 2030 network
- Reduces the resolution of the MSV down to 24H (necessary for faster solve in this example)
- Evaluates project t4 by optimizing the project network (with 1 week rolling horizon)
- Calculates all CBA indicators
- Produces final results in `results/tyndp/NT/cba/`

:::{warning}:::

**Time requirement:** This could take approximately 15-20 minutes to complete, depending on your computer's performance. So this would be a good task to start right before going on a coffee break!


You may have to first undo the changes you've made to your `config/config.tyndp.yaml`:

In [ ]:
! git restore config/config.tyndp.yaml

Checkout to the workshop-specific branch:

In [ ]:
! git checkout workshop-4-cba

Now, you can trigger the run of the complete CBA workflow (using a pre-solved SB network):

In [ ]:
! pixi run tyndp-cba --config 'run={"name":"NT"}' 'cba={"planning_horizons":[2030],"cba_scenario_input":{"use_presolved":true,"sb_version":"latest"},"projects":["t4"],"msv_extraction":{"resolution":"24H"}}'

# Solutions

## Task 2: Reproduce benchmarks

In [ ]:
show_benchmarks(
    "benchmark_hydrogen_price_cy2009",
    [2030],
    "data/results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

In [ ]:
# (a) Try to reproduce the Open-TYNDP outcomes from the hydrogen prices example above from our latest [benchmarking figures](https://zenodo.org/records/20303009).
prices = (
    s2030.prices(bus_carrier="H2", weighting="time").to_frame("Open-TYNDP").sort_index()
)

ax = prices.plot.bar(figsize=(16, 4), ylabel="EUR/MWh_H2", xlabel="Node")

ax.set_facecolor("#e8e8e8")
ax.set_axisbelow(True)
ax.grid(True)

# Legend with average
handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{label} ({prices[label].mean():.1f} EUR/MWh_H2)" for label in labels]
ax.legend(handles, new_labels, loc="upper left")

ax.set_title("Hydrogen Price - Scenario NT - CY 2009 - Year 2030")
ax.tick_params(axis="x", rotation=45)

In [ ]:
# (b) Optional: Try to exclude load shedding from the hydrogen price in 2040.
voll = 3000  # EUR/ MWh_H2
prices = (
    s2040.prices(
        bus_carrier="H2",
        weighting="time",
        groupby_time=False,
    )
    .pipe(lambda x: x.where(x < voll * 0.98))  # Add 2% of numerical tolerance
    .mean(axis=1)
    .to_frame("value")
    .sort_index()
)

ax = prices.plot.bar(figsize=(16, 4), ylabel="EUR/MWh_H2", xlabel="Node")

ax.set_facecolor("#e8e8e8")
ax.set_axisbelow(True)
ax.grid(True)

# Legend with average
handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{label} ({prices[label].mean():.1f} EUR/MWh_H2)" for label in labels]
ax.legend(handles, new_labels, loc="upper left")

ax.set_title("Hydrogen Price excl. Load Shedding - Scenario NT - CY 2009 - Year 2040")
ax.tick_params(axis="x", rotation=45)

## Task 3: Investigate outputs

In [ ]:
# (a) Can you verify the amount of wind generated on the Bornholm Energy Island in 2040 at *10.8 TWh*?
(
    s2040.energy_balance(
        bus_carrier="AC_OH",
        aggregate_across_components=True,
        groupby=["bus", "carrier"],
        components="Generator",
    )
    .div(1e6)  # TWh
    .to_frame("Wind Generation [TWh]")
    .loc["BEIOH01"]
)

In [ ]:
# (b) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

(
    s2040.energy_balance(
        bus_carrier="H2",
        carrier="H2 pipeline",
        aggregate_across_components=True,
        groupby=["carrier", "bus"],
    )
    .div(1e6)
    .sort_values(ascending=False)
    .to_frame("H2 Import (+), H2 Export (-) [TWh]")
    .head()
    .style.format("{:,.2f}")  # Make style a bit prettier
)

In [ ]:
# (c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?
s2040.energy_balance.iplot.area(
    bus_carrier=["AC", "H2"],
    y="value",
    x="snapshot",
    color="carrier",
    stacked=True,
    facet_row="bus_carrier",
    facet_col="country",
    sharex=False,
    sharey=False,
    query="snapshot < '2009-06-08' and snapshot >= '2009-06-01' and country == 'DE'",
    width=1800,
    height=500,
)

We can observe for night of the 6th of June, that wind production drops along with solar electricity production resulting in no hydrogen production via electrolysis for that time. Instead, German hydrogen demand is met via H2 pipeline imports as well as from Cavern Storages and blue Hydrogen production. Pumped hydro, battery storage and electricity imports are utilized to support electricity production in the same period to meet the remaining exogenous electricity demand.

## Task 5: Modify Configuration Files Directly

After modifying `config/scenarios.tyndp.yaml` so that the `NT` scenario is:

```yaml
NT:
  tyndp_scenario: NT

  cba:
    planning_horizons: [2030]
    projects: ["t4"]
```

In [ ]:
# First, run the checkpoint
! pixi run tyndp-checkpoint

In [ ]:
# Run the dry-run without --config arguments
! pixi run tyndp-cba -n

## Task 6: Create a Custom Climate Year Scenario

After creating your custom scenario in `config/scenarios.tyndp.yaml` (e.g., named "NT-workshop"):

In [ ]:
# First, run the checkpoint
! pixi run tyndp-checkpoint --config 'run={"name":"NT-workshop"}'

In [ ]:
# Then run the dry-run to see the workflow
! pixi run tyndp-cba --config 'run={"name":"NT-workshop"}' -n

# Notebook clean up

In [ ]:
# Only clean up data when running in CI environment
if os.getenv("CI"):
    rm_dir = "data/results-0.7.1"
    print(
        f"CI environment detected. Cleaning up notebook data by removing '{rm_dir}' and '{rm_dir}.zip'."
    )
    shutil.rmtree(rm_dir, ignore_errors=True)
    Path(f"{rm_dir}.zip").unlink(missing_ok=True)
else:
    print("Skipping cleanup (not in CI environment).")